# The forward pass

Up until now, we have focused on essential PyTorch syntax, the tools you need to work with tensors, but not yet the deeper architectural ideas required to actually build a model. Remember the overview lecture? In that lecture, we walked through the five steps of deep learning.

Before we continue, let’s refresh your memory with a short exercise.

## Exercise 7 - The five steps of deep learning 

Can you remember the five steps of deep learning?
Write down everything you remember about these steps, from the holistic idea behind each step to the actual math involved.

When you’re done, compare your notes with the overview lecture and see how much you recall.

**Note**
This exercise has no answer in the ExerciseAnswers notebook. You should compare to the overview lecture. 

 ---
## The model's first guess

The forward pass is when the model runs and produces its predictions. The very first time you feed data through the model, the weights and biases are still random because no learning has happened yet. Since the model knows nothing at this point, its first predictions
will also be essentially random.

Even though this first guess is random, it is extremely important. It becomes the starting point for backpropagation and the adjustments of the weights and biases. **This lecture will show how to implement a model’s first guess using only the tensor operations you have learned so far.** In later lectures, you will learn how to use higher‑level libraries for this, but to truly understand the deep‑learning process, we will first build it ourselves.

To do this, we will create a very simple model: linear regression, a structure you will see everywhere when learning about deep neural networks.

The model is defined as:

- y_hat = X @ W +b

Where
- y_hat = our model's prediction
- X = our input data
- W = the weight
- b = the bias 

W and b are the parameters the model can adjust in order to reduce the loss and improve the accuracy of its predictions.

**The entire goal of deep learning is to find the best values for W and b so that the prediction y_hat becomes as close as possible to the true value y (the ground truth).**

 ---
## Exercise 8 - The @ operator

Do you remember what the "@" operator does? Describe it with 2 sentences. 

 ---
## Create data for this lecture 

In [14]:
import torch

# Our batch have 10 samples 
N = 10
# Each sample has 1 input feature and 1 output value 
D_in = 1
D_out = 1

# Create input data
X = torch.randn(N, D_in)

# Create our "ground truth", i.e. the weight and bias
# that will result in a y_hat with the same value as
# our actual y
true_W = torch.tensor([[2.0]]) # true W is 2.0
true_b = torch.tensor(1.0) # true b is 1.0

##!! In real-world applications the model never sees the true W or b.
# We only have input data and the corresponding target labels.
# The model must learn the underlying W and b by comparing its
# predictions to the ground truth and adjusting its parameters
# to minimize the error.

# Create target labels + noise
y_true = X @ true_W + true_b + torch.randn(N, D_out) * 0.1

# Input and ground truth label should be same size
print("X shape:", X.shape)
print("y_true shape:", y_true.shape)

X shape: torch.Size([10, 1])
y_true shape: torch.Size([10, 1])


 ---
## Initialize the model's parameters

Remember: to make PyTorch track a tensor as a learnable parameter, we must wrap it in torch.nn.Parameter. Otherwise it's treated as a normal tensor and will NOT be updated during backpropagation.
(See lecture 02_autograd if you need a refresher!)

In [15]:
# Initialize the model's parameters with random values.
# W has shape (D_in, D_out) so it can be multiplied 
# with X of shape (N, D_in). 
# b has shape (1,) and will be broadcast across all samples.
# Setting requires_grad=True makes PyTorch track these 
# tensors as learnable parameters whose gradients will be 
# computed during backprop.
W = torch.randn(D_in, D_out, requires_grad=True)
b = torch.randn(1, requires_grad=True)

# What you see is the models initial prediction which
# of course will be completly wrong! 
print(f"Initial W: {W}\n")
print(f"Initial b: {b}")


Initial W: tensor([[-0.2675]], requires_grad=True)

Initial b: tensor([1.0291], requires_grad=True)


 ---
## The actual implementation

A linear model has a very simple forward pass:

- y_hat = X @ W + b

This is exactly the same operation we used to generate the synthetic data.
Nothing more happens here, this is the entire forward pass.

**Note:**  
In the code below we define a class for our model.  
You **do not** need to fully understand how this works yet, but the code is heavily commented so you can review it later at your own pace.

A PyTorch model class is built by inheriting from  
**torch.nn.Module**, which is the base class for all neural‑network components.  
Inside the class we create a **linear layer** that automatically manages its own weights and biases.  
The forward() method defines how the model computes its predictions during the  
**forward pass**.

For now, just read the comments and continue, you will understand this structure naturally as we progress. The important thing to notice here is that the model’s prediction **y_hat** looks nothing like the true labels
**y_true** at this stage.


In [16]:
##### !YOU DO NOT NEED TO UNDERSTAND THIS FOR NOW! #####
# We define our own model by creating a class that inherits from
# torch.nn.Module. This is the standard way to build neural networks
# in PyTorch. Every model you ever build will follow this pattern.
class LinearModel(torch.nn.Module):
    def __init__(self):
        # Call the parent constructor. This sets up important internal
        # PyTorch machinery so the model can track parameters, gradients,
        # and be used with optimizers.
        super().__init__()

        # Here we create a single linear layer.
        # torch.nn.Linear automatically creates:
        #   - a weight matrix W of shape (D_in, D_out)
        #   - a bias vector b of shape (D_out,)
        #
        # It also registers these as *parameters*, meaning PyTorch will:
        #   - include them in model.parameters()
        #   - compute gradients for them during backprop
        #   - update them when we use an optimizer
        #
        # In other words: this one line replaces everything we did manually
        # earlier (creating W, b, forward pass, etc).
        self.linear = torch.nn.Linear(D_in, D_out)

    # The forward() method defines how the model computes predictions.
    # When you later write model(X), PyTorch will automatically call
    # this method behind the scenes.
    def forward(self, x):
        # Passing x through the linear layer performs:
        #   y_hat = x @ W + b
        # exactly the same formula we used manually earlier.
        return self.linear(x)
########################################################

# Create an instance of our model class.
# This initializes W and b with random values (just like we did manually),
# but now PyTorch manages them for us.
model = LinearModel()

# For now, we simply reuse our earlier input data X.
# In a real project, X_train would be your training dataset.
X_train = X

# Run the data through the model.
# This triggers the forward() method:
#   y_hat = model(X_train)
# which internally calls:
#   y_hat = self.linear(X_train)
# which internally performs:
#   y_hat = X_train @ W + b
#
# So this single line replaces the entire manual forward pass.
y_hat = model(X_train)

# Print the first 5 predicted values.
# These will be random at first because the model has not been trained yet.
# NOTICE THAT grad_fn IS ACTIVE!! It will say SliceBackward
# since our last operation to y_hat is indexing the five first rows
print(f"Prediction y_hat (only first 5 rows):\n{y_hat[:5]}\n")

# Print the first 5 true labels so we can compare.
print(f"True labels y_true (only first 5 rows): \n{y_true[:5]}")


Prediction y_hat (only first 5 rows):
tensor([[ 0.7959],
        [-1.8269],
        [-0.0277],
        [ 0.0221],
        [-0.3814]], grad_fn=<SliceBackward0>)

True labels y_true (only first 5 rows): 
tensor([[ 3.4892],
        [-2.0793],
        [ 1.7824],
        [ 1.8494],
        [ 1.0758]])


 ---
## Backward pass
Okay, so our model has made its first prediction, and it was completely wrong. That’s expected at this stage. The model now needs to adjust its internal parameters **W** and **b** so that its next prediction gets closer to the true labels **y_true**.

You can think of W and b as two tunable radio knobs, one for volume and one for frequency. Imagine that the target label is a football game being broadcast at a specific frequency. The model turns the knobs slightly and listens to the output after each tiny adjustment. At first, all it hears is static, because the knobs are set randomly. But with each adjustment, the sound becomes a little clearer.

If the path from turning the knob to hearing the sound is the *forward pass*, then the moment the model realizes “this doesn’t sound right” and figures out how to adjust the knobs again is the *backward pass*.

### Loss
The loss is simply a number that tells the model *how wrong* its prediction was. If y_hat is far away from y_true, the loss will be large. If the prediction is close, the loss will be small.

You can think of the loss as the static you hear on the radio. The more static, the worse the signal, and the more the knobs (W and b) need to be adjusted. As the model tunes the knobs correctly, the static decreases,
and the loss goes down. A perfectly tuned signal would mean a loss of zero.

In other words, the backward pass is where PyTorch computes how much each parameter contributed to the error. These gradients tell the model which direction to move **W** and **b** in order to reduce the loss on the next forward pass.

In our case, the loss function, which is the operation that measures how wrong the model’s prediction is, is computed using Mean Squared Error (MSE). MSE looks at the difference between each predicted value and its true label, squares those differences, and then averages them: 

$$
\text{Loss} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2
$$

In other words, for every prediction, find the difference (y_hat-y), square that difference to make it positive (y_hat-y)² and take the average of all those differences.

In [ ]:
# Calculate error
error = y_hat - y_true
# Square error
squared_error = error ** 2
# Take the average
loss = squared_error.mean()

print(f"Loss: {loss}")

Loss: 2.6053578853607178


 ---
## Summary

Our calculated loss does not mean anything on its own, however, this quantification of the wrongness in our model’s prediction is stored at the very end of the computation graph, meaning the loss tensor also has a
grad_fn. This allows the model to trace back through the operations that produced the prediction and understand how each step contributed to the final error. This is the superpower of a neural network: it can compare its output to the true labels, see how wrong it was, and adjust **W** and **b** during each iteration in order to reduce the loss as much as possible.

**Note:**  
The whole idea of a neural network is to decrease the loss as much as possible until the training eventually stagnates, at that point, the training (the tuning of **W** and **b**) is finalized.

In the next lecture we will learn how the trining process knows which direction to turn the knobs W and b. 